# Multi-Stage Progress Example with SSE

> Demonstrates a multi-stage data processing pipeline with hierarchical progress tracking using Server-Sent Events (SSE), featuring real-time stage indicators, nested tqdm progress bars with configurable throttling, and HTMX SSE-powered UI state management for concurrent job control.

In [1]:
from fasthtml.common import *
from fasthtml.common import sse_message
import uuid, time, json
import random
import asyncio

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream_async

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_sizes, badge_styles, badge_colors
from cjm_fasthtml_daisyui.components.data_input.range_slider import range_dui, range_colors
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.layout import display_tw
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_align
from cjm_fasthtml_tailwind.utilities.sizing import w, h, container
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_wrap, gap
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.CYBERPUNK)
app.hdrs.append(Link(rel='icon', type='image/svg+xml', href=f'https://api.dicebear.com/9.x/adventurer/svg?seed={random.random()}'))

# Initialize
monitor = ProgressMonitor(keep_history=True)
runner = JobRunner(monitor)

# Storage for results, active jobs, and metadata
job_results = {}
active_jobs = set()
job_metadata = {}  # Store job type and configuration

In [4]:
# Add HTMX SSE extension after HTMX script
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

htmx_idx = get_htmx_idx(app.hdrs)
if htmx_idx >= 0:
    # Insert SSE extension right after HTMX
    app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse"))
    app.hdrs.insert(htmx_idx+2, Script(code="""window.addEventListener('beforeunload',()=>{document.querySelectorAll('[sse-connect]').forEach(e=>{e['htmx-internal-data']?.sseEventSource?.close()});htmx?.findAll('[sse-connect]').forEach(e=>htmx.trigger(e,'htmx:sseClose'))});"""))
else:
    print("HTMX not found")

In [5]:
# Complex multi-stage pipeline
def data_pipeline(dataset_size=1000):
    from tqdm import tqdm
    import time
    
    results = {
        "stages": [],
        "metrics": {}
    }
    
    # Stage 1: Data Collection
    print("Starting Stage 1: Data Collection")
    collected_data = []
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source A"):
        time.sleep(0.002)
        collected_data.append(f"source_a_{i}")
    
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source B"):
        time.sleep(0.002)
        collected_data.append(f"source_b_{i}")
    
    results["stages"].append("Data Collection Complete")
    results["metrics"]["collected"] = len(collected_data)
    
    # Stage 2: Data Validation
    print("Starting Stage 2: Data Validation")
    valid_data = []
    for item in tqdm(collected_data, desc="2. Validating Data"):
        time.sleep(0.001)
        if random.random() > 0.1:  # 90% pass validation
            valid_data.append(item)
    
    results["stages"].append("Data Validation Complete")
    results["metrics"]["validated"] = len(valid_data)
    
    # Stage 3: Data Transformation
    print("Starting Stage 3: Data Transformation")
    
    # Sub-stage 3.1: Normalization
    normalized = []
    for item in tqdm(valid_data[:len(valid_data)//2], desc="3.1 Normalizing"):
        time.sleep(0.002)
        normalized.append(f"norm_{item}")
    
    # Sub-stage 3.2: Encoding
    encoded = []
    for item in tqdm(valid_data[len(valid_data)//2:], desc="3.2 Encoding"):
        time.sleep(0.002)
        encoded.append(f"enc_{item}")
    
    # Sub-stage 3.3: Merging
    merged = []
    for i in tqdm(range(min(len(normalized), len(encoded))), desc="3.3 Merging"):
        time.sleep(0.001)
        merged.append((normalized[i], encoded[i]))
    
    results["stages"].append("Data Transformation Complete")
    results["metrics"]["transformed"] = len(merged)
    
    # Stage 4: Analysis
    print("Starting Stage 4: Analysis")
    
    # Parallel analysis tasks
    for i in tqdm(range(100), desc="4.1 Statistical Analysis"):
        time.sleep(0.01)
    
    for i in tqdm(range(80), desc="4.2 Pattern Detection"):
        time.sleep(0.012)
    
    for i in tqdm(range(60), desc="4.3 Anomaly Detection"):
        time.sleep(0.015)
    
    results["stages"].append("Analysis Complete")
    results["metrics"]["patterns_found"] = random.randint(10, 50)
    results["metrics"]["anomalies"] = random.randint(1, 10)
    
    # Stage 5: Report Generation
    print("Starting Stage 5: Report Generation")
    
    for i in tqdm(range(50), desc="5.1 Generating Charts"):
        time.sleep(0.02)
    
    for i in tqdm(range(30), desc="5.2 Writing Summary"):
        time.sleep(0.03)
    
    for i in tqdm(range(20), desc="5.3 Finalizing Report"):
        time.sleep(0.04)
    
    results["stages"].append("Report Generation Complete")
    results["report"] = "pipeline_report.pdf"
    
    return results

In [6]:
# Nested loop example
def nested_processing():
    from tqdm import tqdm
    import time
    
    results = []
    
    # Outer loop - batches
    batches = 5
    items_per_batch = 20
    
    for batch in tqdm(range(batches), desc="Processing Batches"):
        batch_results = []
        
        # Inner loop - items in batch
        for item in tqdm(range(items_per_batch), desc=f"Batch {batch+1} Items", leave=False):
            time.sleep(0.05)
            batch_results.append(f"batch_{batch}_item_{item}")
        
        results.append(batch_results)
        time.sleep(0.1)  # Pause between batches
    
    return results

In [7]:
# Helper functions for UI rendering
def render_task_buttons(disabled=False, oob_swap=False):
    """Render task buttons with appropriate state"""
    btn_classes_pipeline = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    btn_classes_nested = combine_classes(
        btn, 
        btn_colors.secondary,
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'id': 'task-buttons',
    }
    if oob_swap:
        kwargs['hx_swap_oob'] = 'true'
    
    return Div(
        Form(
            Input(type="hidden", name="task_type", value="pipeline"),
            Input(type="hidden", name="dataset_size", id="dataset-size-hidden", value="1000"),
            Button(
                "Start Pipeline", 
                type="submit",
                id="start-pipeline-btn",
                cls=btn_classes_pipeline,
                disabled=disabled,
                onclick="document.getElementById('dataset-size-hidden').value = document.getElementById('dataset-size').value" if not disabled else None
            ),
            hx_post="/start_task" if not disabled else None,
            hx_target="#progress-container",
            hx_swap="innerHTML",
            cls=combine_classes(display_tw.inline_block, m.r(0.5))
        ),
        Form(
            Input(type="hidden", name="task_type", value="nested"),
            Button(
                "Run Nested Task",
                type="submit",
                id="run-nested-btn",
                cls=btn_classes_nested,
                disabled=disabled
            ),
            hx_post="/start_task" if not disabled else None,
            hx_target="#progress-container",
            hx_swap="innerHTML",
            cls=combine_classes(display_tw.inline_block, m.r(0.5))
        ),
        Button(
            "Clear Results",
            hx_post="/clear_results",
            hx_target="#results",
            hx_swap="outerHTML",
            cls=combine_classes(btn, btn_colors.warning)
        ),
        **kwargs
    )

In [8]:
# Helper functions to reduce code duplication

# SSE streaming helpers
async def parse_sse_data(data):
    """Parse SSE data and return progress data if valid"""
    if data.startswith('data: '):
        try:
            json_str = data[6:].strip()
            if json_str and json_str != '{}':
                return json.loads(json_str)
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
    return None

async def sse_job_stream(job_id, handler_fn, interval=0.1):
    """
    Generic SSE streaming handler for job monitoring.
    
    Args:
        job_id: The job ID to monitor
        handler_fn: Async function(progress_data, is_completed) that yields SSE messages
        interval: Update interval for streaming
    """
    try:
        async for data in sse_stream_async(
            monitor,
            job_id,
            interval=interval,
            heartbeat=30.0,
            wait_for_start=True,
            start_timeout=5.0
        ):
            if data.startswith(': '):
                yield data  # Heartbeat
                continue
                
            progress_data = await parse_sse_data(data)
            if progress_data:
                is_completed = progress_data.get('completed', False)
                
                # Call the handler function
                async for message in handler_fn(progress_data, is_completed):
                    yield message
                
                if is_completed:
                    break
                    
    except Exception as e:
        print(f"Error in SSE stream for job {job_id[:8]}: {e}")
        import traceback
        traceback.print_exc()

# UI rendering helpers
def render_progress_bar(value, max_value="100", color=None, width=None, height=None, bar_id=None):
    """Render a progress bar with optional styling"""
    classes = [progress]
    if color:
        classes.append(color)
    if width:
        classes.append(width)
    if height:
        classes.append(height)
    
    kwargs = {
        'value': str(value),
        'max': max_value,
        'cls': combine_classes(*classes)
    }
    if bar_id:
        kwargs['id'] = bar_id
    
    return Progress(**kwargs)

def render_overall_progress_div(progress_value, include_wrapper_id=False):
    """Render the overall progress section"""
    kwargs = {}
    if include_wrapper_id:
        kwargs['id'] = 'overall-progress-wrapper'
        kwargs['hx_swap_oob'] = 'true'
    
    return Div(
        P("Overall Progress:", cls=str(font_weight.bold)),
        render_progress_bar(
            progress_value, 
            color=progress_colors.primary, 
            width=w.full, 
            height=h(6),
            bar_id="overall-progress"
        ),
        P(f"{progress_value:.1f}%" if isinstance(progress_value, (int, float)) else f"{progress_value}%", 
          cls=combine_classes(text_align.center, m.t(1)), 
          id="overall-percent"),
        **kwargs
    )

def render_task_progress_bar(bar_id, bar_data, bar_color=None):
    """Render a single task progress bar"""
    if not bar_color:
        bar_color = get_bar_color(bar_data.description)
    
    return Div(
        P(f"{bar_data.description}: {bar_data.progress:.1f}% ({bar_data.current}/{bar_data.total})", 
          cls=combine_classes(font_size.sm, font_weight.semibold, m.b(1))),
        render_progress_bar(
            bar_data.progress,
            color=bar_color,
            width=w.full
        ),
        cls=combine_classes(p(3), bg_dui.base_300, rounded()),
        id=f"task-{bar_id}"
    )

def render_active_tasks_container(bars_html, title="Active Tasks:", completed=False):
    """Render the active tasks container"""
    elements = [
        H3(title, cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
        Div(*bars_html, id="active-tasks", cls=str(space.y(3)))
    ]
    
    if completed:
        elements.append(
            H3("Completed!", cls=combine_classes(font_weight.semibold, m.t(6), text_dui.success))
        )
    
    return Div(*elements, id="tasks-container", hx_swap_oob="true")

def should_enable_buttons():
    """Check if buttons should be enabled (no active jobs)"""
    return len(active_jobs) == 0

def render_final_results(job_id):
    """Render the final results pre element"""
    results = job_results.get(job_id, {})
    return Pre(
        json.dumps(results, indent=2),
        id="results",
        cls=combine_classes(bg_dui.base_300, p(4), rounded()),
        hx_swap_oob="true"
    )

In [9]:
# Job management helpers
def create_job_wrapper(job_id, job_fn, *args, **kwargs):
    """Create a wrapper function for job execution"""
    def wrapper():
        try:
            result = job_fn(*args, **kwargs)
            job_results[job_id] = result
        finally:
            # Always clean up active jobs set
            active_jobs.discard(job_id)
            # Note: We keep metadata and results for display
    return wrapper

def render_progress_ui(job_id, task_type):
    """Render the progress UI with SSE connections"""
    is_pipeline = task_type == 'pipeline'
    stages = ['Collection', 'Validation', 'Transformation', 'Analysis', 'Report'] if is_pipeline else []
    
    return Div(
        # Disable buttons via OOB swap
        render_task_buttons(disabled=True, oob_swap=True),
        # Progress display with SSE
        Div(
            H2("Pipeline Progress", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
            # Stage indicators (if pipeline)
            Div(
                *[render_stage_indicator(stage, 'pending') for stage in stages],
                cls=combine_classes(flex_display, flex_wrap.wrap, gap(2), m.b(4)),
                id="stage-indicators",
                hx_ext="sse" if stages else None,
                sse_connect=f"/stream_stages?job_id={job_id}" if stages else None,
                sse_swap="message" if stages else None
            ) if stages else None,
            # Overall progress with SSE
            Div(
                render_overall_progress_div(0),
                id="overall-progress-wrapper",
                hx_ext="sse",
                sse_connect=f"/stream_overall_progress?job_id={job_id}",
                sse_swap="message"
            ),
            # Active tasks container with SSE
            Div(
                H3("Starting...", cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
                Div(
                    id="active-tasks",
                    cls=str(space.y(3))
                ),
                id="tasks-container",
                hx_ext="sse",
                sse_connect=f"/stream_tasks?job_id={job_id}",
                sse_swap="message"
            ),
            cls=combine_classes(card, bg_dui.base_200, p(6))
        )
    )

In [10]:
def render_stage_indicator(stage, status='pending'):
    """Render a single stage indicator badge"""
    if status == 'complete':
        badge_class = combine_classes(badge, badge_sizes.lg, badge_colors.success)
    elif status == 'active':
        badge_class = combine_classes(badge, badge_sizes.lg, badge_colors.warning)
    else:
        badge_class = combine_classes(badge, badge_sizes.lg, badge_styles.outline)
    
    return Span(stage, cls=badge_class, id=f"stage-{stage.lower().replace(' ', '-')}")

In [11]:
def detect_active_stages(bars):
    """Detect active and completed stages from progress bars"""
    stages = {
        'Collection': 'pending',
        'Validation': 'pending', 
        'Transformation': 'pending',
        'Analysis': 'pending',
        'Report': 'pending'
    }
    
    # Track which stages we've seen
    seen_stages = set()
    
    for bar_id, bar in bars.items():
        desc = bar.description or ""
        progress = bar.progress
        
        if '1.' in desc or 'Collecting' in desc:
            seen_stages.add('Collection')
            stages['Collection'] = 'complete' if progress >= 100 else 'active'
        elif '2.' in desc or 'Validating' in desc:
            seen_stages.add('Validation')
            stages['Validation'] = 'complete' if progress >= 100 else 'active'
            if 'Collection' not in seen_stages:
                stages['Collection'] = 'complete'
        elif '3.' in desc or 'Transform' in desc or 'Normaliz' in desc or 'Encod' in desc or 'Merg' in desc:
            seen_stages.add('Transformation')
            stages['Transformation'] = 'complete' if progress >= 100 else 'active'
            if 'Validation' not in seen_stages:
                stages['Validation'] = 'complete'
            if 'Collection' not in seen_stages:
                stages['Collection'] = 'complete'
        elif '4.' in desc or 'Analysis' in desc or 'Pattern' in desc or 'Anomaly' in desc:
            seen_stages.add('Analysis')
            stages['Analysis'] = 'complete' if progress >= 100 else 'active'
            for prev_stage in ['Collection', 'Validation', 'Transformation']:
                if prev_stage not in seen_stages:
                    stages[prev_stage] = 'complete'
        elif '5.' in desc or 'Report' in desc or 'Chart' in desc or 'Summary' in desc:
            seen_stages.add('Report')
            stages['Report'] = 'complete' if progress >= 100 else 'active'
            for prev_stage in ['Collection', 'Validation', 'Transformation', 'Analysis']:
                if prev_stage not in seen_stages:
                    stages[prev_stage] = 'complete'
    
    return stages

In [12]:
def get_bar_color(description):
    """Get progress bar color based on description"""
    if not description:
        return progress_colors.accent
    
    if '1.' in description:
        return progress_colors.info
    elif '2.' in description:
        return progress_colors.success
    elif '3.' in description:
        return progress_colors.warning
    elif '4.' in description:
        return progress_colors.error
    elif '5.' in description:
        return progress_colors.primary
    else:
        return progress_colors.accent

In [13]:
# Main page
@rt("/")
def index():
    # Check if there are any active jobs
    has_active = len(active_jobs) > 0
    
    return create_test_page(
        "Multi-Stage Progress Demo - SSE",
        Div(
            H1("Multi-Stage Pipeline Progress (SSE)", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Pipeline controls
            Div(
                H2("Pipeline Configuration", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Label("Dataset Size:", cls=str(label)),
                    Input(
                        id="dataset-size",
                        name="dataset_size",
                        type="range",
                        min="100",
                        max="2000",
                        value="1000",
                        cls=combine_classes(range_dui, range_colors.primary),
                        oninput="document.getElementById('size-display').textContent = this.value"
                    ),
                    Span("1000", id="size-display", cls=combine_classes(badge, badge_sizes.lg, m.l(2))),
                    cls=combine_classes(m.b(4))
                ),
                render_task_buttons(disabled=has_active),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress container - will check for active jobs on load
            Div(
                id="progress-container",
                hx_get="/check_active_jobs",
                hx_trigger="load",
                hx_swap="innerHTML",
                cls=str(m.b(6))
            ),
            
            # Results
            Div(
                H2("Results", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Pre(
                    json.dumps(job_results, indent=2) if job_results else "No results yet",
                    id="results",
                    cls=combine_classes(bg_dui.base_300, p(4), rounded())
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [14]:
@rt("/start_task")
async def start_task(request):
    form = await request.form()
    task_type = form.get('task_type', 'pipeline')
    dataset_size = int(form.get('dataset_size', 1000))
    
    job_id = str(uuid.uuid4())
    active_jobs.add(job_id)
    
    # Store job metadata
    job_metadata[job_id] = {
        'type': task_type,
        'dataset_size': dataset_size,
        'started_at': time.time()
    }
    
    # Configure job based on type
    if task_type == 'nested':
        wrapper = create_job_wrapper(
            job_id, 
            nested_processing
        )
        patch_kwargs = {
            "min_delta_pct": 5,
            "min_update_interval": 0.1,
            "emit_initial": True
        }
    else:
        wrapper = create_job_wrapper(
            job_id,
            data_pipeline,
            dataset_size
        )
        patch_kwargs = {
            "min_delta_pct": 1,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    
    # Start the job
    runner.start(job_id, wrapper, patch_kwargs=patch_kwargs)
    
    # Return progress UI with SSE connections
    return render_progress_ui(job_id, task_type)

In [15]:
@rt("/check_active_jobs")
def check_active_jobs():
    """Check for active jobs on page load and render their progress UI"""
    
    # If no active jobs, return empty
    if not active_jobs:
        return ""
    
    # Get the first active job (in a real app you might handle multiple)
    job_id = next(iter(active_jobs))
    
    # Get job metadata to determine type
    meta = job_metadata.get(job_id, {})
    task_type = meta.get('type', 'pipeline')
    
    # Check if job is still actually running in monitor
    snapshot = monitor.snapshot(job_id)
    if not snapshot or snapshot.get('completed', False):
        # Job finished but wasn't cleaned up properly
        active_jobs.discard(job_id)
        return ""
    
    # Return the progress UI with SSE connections for the active job
    return render_progress_ui(job_id, task_type)

In [16]:
@rt("/clear_results")
def clear_results():
    """Clear completed job results and metadata"""
    # Clear results for non-active jobs
    for job_id in list(job_results.keys()):
        if job_id not in active_jobs:
            del job_results[job_id]
            if job_id in job_metadata:
                del job_metadata[job_id]
    
    # Return updated results display
    return Pre(
        "No results yet",
        id="results",
        cls=combine_classes(bg_dui.base_300, p(4), rounded()),
        hx_swap_oob="true"
    )

In [17]:
# SSE endpoint for overall progress
@rt("/stream_overall_progress")
def stream_overall_progress(job_id: str):
    """SSE endpoint for streaming overall progress updates"""
    
    async def handle_progress(progress_data, is_completed):
        if is_completed:
            # Job completed - send final updates with OOB swaps
            elements = [
                render_overall_progress_div(100, include_wrapper_id=True)
            ]
            
            # Re-enable buttons if no other active jobs
            if should_enable_buttons():
                elements.append(render_task_buttons(disabled=False, oob_swap=True))
            
            # Update results
            elements.append(render_final_results(job_id))
            
            yield sse_message(Div(*elements))
        else:
            # Update progress
            progress_value = progress_data.get('progress', 0)
            yield sse_message(render_overall_progress_div(progress_value))
    
    return EventStream(sse_job_stream(job_id, handle_progress, interval=0.1))

In [18]:
# SSE endpoint for active tasks
@rt("/stream_tasks")
def stream_tasks(job_id: str):
    """SSE endpoint for streaming active task progress bars"""
    
    last_bars = {}
    final_bars = []
    
    async def handle_tasks(progress_data, is_completed):
        nonlocal last_bars, final_bars
        
        if is_completed:
            # Fetch fresh snapshot for final state
            await asyncio.sleep(0.1)
            final_snapshot = monitor.snapshot(job_id)
            
            if final_snapshot and final_snapshot['bars']:
                final_bars = []
                for bar_id, bar in final_snapshot['bars'].items():
                    final_bars.append(render_task_progress_bar(bar_id, bar))
            
            # Send final state with completion message
            yield sse_message(
                render_active_tasks_container(final_bars, completed=True)
            )
        else:
            # Get current bars from snapshot
            snapshot = monitor.snapshot(job_id)
            if snapshot and snapshot['bars']:
                bars_html = []
                current_bar_ids = set()
                
                for bar_id, bar in snapshot['bars'].items():
                    current_bar_ids.add(bar_id)
                    bars_html.append(render_task_progress_bar(bar_id, bar))
                    final_bars = bars_html.copy()
                
                # Check for bar changes (for debugging)
                if current_bar_ids != set(last_bars.keys()):
                    last_bars = {bid: True for bid in current_bar_ids}
                
                # Send update
                yield sse_message(
                    Div(
                        H3("Active Tasks:", cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
                        Div(*bars_html, id="active-tasks", cls=str(space.y(3)))
                    )
                )
    
    return EventStream(sse_job_stream(job_id, handle_tasks, interval=0.1))

In [19]:
# SSE endpoint for stage indicators
@rt("/stream_stages")
def stream_stages(job_id: str):
    """SSE endpoint for streaming stage indicator updates"""
    
    last_stages = {}
    
    async def handle_stages(progress_data, is_completed):
        nonlocal last_stages
        
        stages_list = ['Collection', 'Validation', 'Transformation', 'Analysis', 'Report']
        
        if is_completed:
            # Mark all stages as complete
            yield sse_message(
                Div(
                    *[render_stage_indicator(stage, 'complete') for stage in stages_list],
                    cls=combine_classes(flex_display, flex_wrap.wrap, gap(2), m.b(4)),
                    id="stage-indicators",
                    hx_swap_oob="true"
                )
            )
        else:
            # Get current stage statuses
            snapshot = monitor.snapshot(job_id)
            if snapshot and snapshot['bars']:
                current_stages = detect_active_stages(snapshot['bars'])
                
                # Check if stages have changed
                if current_stages != last_stages:
                    last_stages = current_stages.copy()
                    
                    # Send updated stage indicators
                    stage_elements = []
                    for stage, status in current_stages.items():
                        stage_elements.append(render_stage_indicator(stage, status))
                    
                    yield sse_message(
                        Div(
                            *stage_elements,
                            cls=combine_classes(flex_display, flex_wrap.wrap, gap(2), m.b(4))
                        )
                    )
    
    return EventStream(sse_job_stream(job_id, handle_stages, interval=0.1))

In [20]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [21]:
# Stop server when done
server.stop()